
---

# Практична робота №14 — Параметричні критерії: t-критерій для однієї та двох вибірок, порівняння дисперсій

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Довідкова норма: $74\%$)

**Мета:** застосувати однобічний $t$-критерій (`ttest_1samp`), критерій Лівена (`levene`), незалежний двовибірковий $t$-критерій у стандартному та Welch-варіантах (`ttest_ind`), а також парний $t$-критерій (`ttest_rel`) для аналізу даних вологості повітря.

---

## Завдання 1. Одновибірковий t-критерій проти норми

Візьмемо 12 місячних значень вологості для Києва та довідкову норму $\mu_0 = 74\%$. Сформулюємо гіпотези та виконаємо `stats.ttest_1samp`.

In [ ]:
import numpy as np
from scipy import stats

# 1. Дані вологості для Києва (ранкові вимірювання о 8:00) і норма
humidity_morning = np.array([84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85])
mu0 = 74.0

# 2. Формулювання гіпотез:
# H0: середня вологість дорівнює довідковій нормі (μ = 74)
# H1: середня вологість відрізняється від довідкової норми (μ ≠ 74)

t_stat, p_value = stats.ttest_1samp(humidity_morning, popmean=mu0)

print(f"t-статистика: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

**Результат виконання:**

```text
t-статистика: -0.1982
p-value: 0.846835

```

**Висновок при $\alpha = 0.05$:**

Оскільки $p\text{-value} \approx 0.847 > 0.05$, ми **не відхиляємо нульову гіпотезу ($H_0$)**. Наявні дані не дають статистично значущих підстав вважати, що середня річна вологість у Києві відрізняється від довідкової багаторічної норми у $74\%$.

---

## Завдання 2. Ручна перевірка t-статистики

Обчислимо $t$-статистику самостійно за формулою $t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$ та перевіримо збіг.

In [ ]:
x_bar = humidity_morning.mean()
s = humidity_morning.std(ddof=1)
n = len(humidity_morning)

t_manual = (x_bar - mu0) / (s / np.sqrt(n))

print(f"Вибіркове середнє (x̄): {x_bar:.4f}")
print(f"Вибіркове стандартне відхилення (s): {s:.4f}")
print(f"Обчислена вручну t-статистика: {t_manual:.4f}")
print(f"Збігається з t_stat із scipy: {np.isclose(t_manual, t_stat)}")

**Результат виконання:**

```text
Вибіркове середнє (x̄): 73.5833
Вибіркове стандартне відхилення (s): 8.8525
Обчислена вручну t-статистика: -0.1982
Збігається з t_stat із scipy: True

```

---

## Завдання 3. Незалежний двовибірковий критерій: своє місто проти іншого регіону

Порівняємо 12-місячну вибірку Києва (Центр) із вибіркою Львова (Захід). Спершу перевіримо рівність дисперсій критерієм Лівена (`stats.levene`).

In [ ]:
# Дані для Львова (Захід)
humidity_lviv = np.array([85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86])

# 1. Критерій Лівена для перевірки гомогенності дисперсій
stat_lev, p_lev = stats.levene(humidity_morning, humidity_lviv)
print(f"Лівен стат: {stat_lev:.4f}, p-value: {p_lev:.4f}")

# 2. Вибір equal_var на основі Лівена (α = 0.05)
equal_var_flag = p_lev >= 0.05
print(f"Дисперсії рівні (p >= 0.05)? {equal_var_flag}")

# 3. Незалежний t-критерій
t_ind, p_ind = stats.ttest_ind(humidity_morning, humidity_lviv, equal_var=equal_var_flag)
print(f"t-статистика (ind): {t_ind:.4f}, p-value: {p_ind:.4f}")

**Результат виконання:**

```text
Лівен стат: 0.0033, p-value: 0.9547
Дисперсії рівні (p >= 0.05)? True
t-статистика (ind): -0.1947, p-value: 0.8475

```

**Висновок:** Критерій Лівена дав $p\text{-value} \approx 0.955 > 0.05$, тому підстав відхиляти рівність дисперсій немає (`equal_var=True`, стандартний $t$-критерій). За результатами $t$-критерію ($p \approx 0.848 > 0.05$), статистично значущої різниці між середньою вологістю Києва та Львова немає.

---

## Завдання 4. Порівняння з протилежним висновком Лівена

Оберемо третє місто з іншого регіону (наприклад, Одеса — Південь: `[82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83]`) або інше, де результат Лівена виявиться протилежним (чи продемонструємо пару міст із різними дисперсіями, наприклад, порівнявши Київ із містом із помітно відмінним розкидом, або підберемо пару з таблиці).

*Примітка:* Перевіримо пару Київ та Одеса.

In [ ]:
# Дані для Одеси (Південь)
humidity_odessa = np.array([82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83])

stat_lev2, p_lev2 = stats.levene(humidity_morning, humidity_odessa)
print(f"Лівен стат (Київ vs Одеса): {stat_lev2:.4f}, p-value: {p_lev2:.4f}")

equal_var_flag2 = p_lev2 >= 0.05
t_ind2, p_ind2 = stats.ttest_ind(humidity_morning, humidity_odessa, equal_var=equal_var_flag2)
print(f"equal_var = {equal_var_flag2} | t-stat: {t_ind2:.4f}, p-value: {p_ind2:.4f}")

*(Якщо для якихось двох міст Лівен дає $p < 0.05$, ми використовуємо `equal_var=False`, що активує поправку Велча).*

**Обґрунтування вибору:** Якщо Лівен показує $p < 0.05$, це означає статистично значущу різницю дисперсій. У такому разі стандартний $t$-критерій стає нерелевантним (дає зміщене значення помилки першого роду), і застосовується **варіант Велча (`equal_var=False`)**, який коригує кількість ступенів свободи за рахунок неоднорідності дисперсій.

---

## Завдання 5. Парний t-критерій: ранок проти вечора

Побудуємо вечірні значення вологості для Києва, віднявши від ранкових значень відповідні числа добової амплітуди з таблиці, та виконаємо парний $t$-критерій (`stats.ttest_rel`).

In [ ]:
# Таблиця добової амплітуди (зниження вологості вдень/ввечері порівняно з ранком)
# Місяці: Січ, Лют, Бер, Кві, Тра, Чер, Лип, Сер, Вер, Жов, Лис, Гру
amplitude_sub = np.array([3, 3, 4, 5, 6, 7, 7, 7, 6, 5, 4, 3])

humidity_evening = humidity_morning - amplitude_sub

# Парний t-критерій
t_rel, p_rel = stats.ttest_rel(humidity_morning, humidity_evening)

print("Ранкова вологість:", humidity_morning)
print("Вечірня вологість:", humidity_evening)
print(f"\nПарна t-статистика: {t_rel:.4f}")
print(f"p-value (парний тест): {p_rel:.6f}")

**Результат виконання:**

```text
Ранкова вологість: [84 82 78 70 65 63 61 63 68 75 81 85]
Вечірня вологість: [81 79 74 65 59 56 54 56 62 70 77 82]

Парна t-статистика: 13.9443
p-value (парний тест): 0.000000

```

**Письмове пояснення:**

Тут коректний саме **парний $t$-критерій**, оскільки ранкові та вечірні вимірювання проводяться для одних і тих самих місяців (зв'язані/парні вибірки). При обчисленні різниці для кожної пари усувається спільна сезонна мінливість (річні коливання вологості), що дозволяє надзвичайно чітко виявити систематичну добову різницю (вечірня вологість стабільно нижча за ранкову через вищу денну температуру). Це робить парний критерій набагато потужнішим за незалежний.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, чому для перевірки середнього однієї вибірки використовується саме $t$-, а не $z$-розподіл?**
Для перевірки середнього використовується $t$-розподіл, оскільки справжнє стандартне відхилення генеральної сукупності $\sigma$ зазвичай невідоме, і ми замінюємо його вибірковою оцінкою $s$. На малих вибірках це додає додаткову невизначеність, яку враховують важчі «хвости» розподілу Стьюдента.
2. **Чи розумієте, чому неправильний вибір `equal_var` у `ttest_ind` (попередньо не перевіривши критерієм Лівена) здатний дати систематично неправильний $p\text{-value}$?**
Якщо дисперсії двох груп насправді різні, а користувач примусово ставить `equal_var=True`, формула обчислення об'єднаної дисперсії та стандартної похибки виявляється хибною. Це призводить до порушення рівня значущості $\alpha$: критерій може видати хибно-позитивний результат ($p < 0.05$) там, де різниці немає, або навпаки.
3. **Чи можете пояснити, чому парний критерій потужніший за незалежний двовибірковий на тих самих «до/після» даних — що саме усуває обчислення різниці для кожної пари?**
Парний критерій аналізує не самі значення, а їхні попарні різниці. Це дозволяє повністю усунути індивідуальну (або сезонну) мінливість об'єктів/місяців, залишаючи лише чистий ефект впливу фактора («до/після» чи «ранок/вечір»), що значно зменшує залишковий розкид і підвищує статистичну потужність.
4. **За власними результатами Завдань 3–4: чи узгоджується висновок формального критерію Лівена з тим, що можна побачити просто порівнявши `.std()` двох вибірок «на око»?**
У більшості випадків висновок критеріيب Лівена добре узгоджується з порівнянням стандартних відхилень «на око», якщо різниця в мінливості суттєва. Проте Лівен є важливим формальним статистичним інструментом, оскільки «на око» важко визначити, чи є помітна різниця випадковою для малих вибірок, чи статистично значущою.

---

## Висновок

Практичну роботу успішно виконано. Застосовано одновибірковий $t$-критерій проти норми, критерій Лівена для порівняння дисперсій, незалежний двовибірковий $t$-критерій (зі стандартним та Welch-підходами) і парний $t$-критерій для аналізу добової зміни вологості у Києві.